### Análise Exploratória das Métricas Globais de Conectividade

Este notebook apresenta a análise exploratória das métricas globais de conectividade funcional cerebral extraídas a partir de sinais de EEG em estado de repouso.

O objetivo desta etapa é:
- caracterizar estatisticamente as métricas globais,
- investigar diferenças entre condições experimentais (EO vs EC),
- fornecer subsídios para as etapas posteriores de redução de dimensionalidade e aprendizado de máquina.


### Import e carregamento

In [7]:
import pandas as pd
import numpy as np
import plotly.io as pio

import plotly.express as px
import plotly.graph_objects as go


In [8]:
pio.templates.default = "plotly_dark"

In [9]:
# Carrega dataset final
df_dataset = pd.read_parquet("../../data/processed/dataset_features.parquet")

df_dataset.shape

(108, 409)

In [10]:
df_dataset

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
0,14786.0,14223.0,13305.0,12562.0,9459.0,13334.0,11893.0,8054.0,8195.0,15173.0,...,False,False,False,False,True,False,True,True,False,False
1,20374.0,20232.0,24459.0,20399.0,15591.0,16955.0,16395.0,15200.0,13285.0,22416.0,...,False,False,False,False,True,False,True,True,False,False
2,22982.0,24351.0,22231.0,17394.0,14843.0,20348.0,21595.0,21675.0,12499.0,16804.0,...,False,False,False,False,True,False,True,False,True,True
3,8682.0,9045.0,16592.0,12541.0,8066.0,11202.0,15684.0,15513.0,8463.0,10332.0,...,False,False,False,False,True,False,True,False,True,True
4,29168.0,28481.0,26967.0,27563.0,27400.0,24913.0,23154.0,24220.0,20045.0,14112.0,...,False,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,12897.0,10346.0,15237.0,15322.0,11459.0,13496.0,11976.0,17013.0,14137.0,13124.0,...,False,False,False,False,True,True,False,True,False,True
104,21065.0,16982.0,22893.0,21942.0,16863.0,23027.0,17824.0,24175.0,19089.0,15923.0,...,False,False,False,False,True,False,True,True,False,False
105,21880.0,19799.0,24711.0,16700.0,10544.0,16339.0,19149.0,25439.0,12665.0,14468.0,...,False,False,False,False,True,False,True,True,False,False
106,21049.0,18595.0,23879.0,19529.0,11742.0,13766.0,19801.0,22879.0,12783.0,10114.0,...,False,False,False,False,True,False,True,True,False,True


In [11]:
df_dataset['condition'].value_counts()

condition
EC    54
EO    54
Name: count, dtype: int64

### Seleção das métricas globais

“As métricas globais representam propriedades agregadas da rede funcional cerebral, independententes de regiões específicas, refletindo o nível geral de integração e variabilidade da conectividade.”

In [13]:
GLOBAL_FEATURES = [
    "global_efficiency",
    "edges_mean",
    "edges_std",
    "edges_cv"
]

id_cols = ["subject_id", "condition"]

df_global = df_dataset[id_cols + GLOBAL_FEATURES]

df_global.head()

,subject_id,condition,global_efficiency,edges_mean,edges_std,edges_cv
0,sub-010069,EC,1.0,403.200493,154.681917,0.383635
1,sub-010069,EO,1.0,490.755957,194.198347,0.395713
2,sub-010070,EC,1.0,547.977814,187.201631,0.341623
3,sub-010070,EO,1.0,439.951520,216.767714,0.492708
4,sub-010079,EC,1.0,634.027938,185.903906,0.293211


In [14]:
df_global[GLOBAL_FEATURES].describe()

,global_efficiency,edges_mean,edges_std,edges_cv
count,108.0,108.000000,108.000000,108.000000
mean,1.0,586.454776,195.914256,0.336977
std,0.0,131.515415,50.942458,0.064164
min,1.0,316.432210,118.619167,0.222316
25%,1.0,510.374486,156.474458,0.289174
50%,1.0,575.940016,190.981762,0.333624
75%,1.0,642.412490,225.767922,0.372625
max,1.0,1161.838127,341.371652,0.537939


In [16]:
df_global.groupby("condition")[GLOBAL_FEATURES].describe()


global_efficiency                                    edges_mean  \
                      count mean  std  min  25%  50%  75%  max      count   
condition                                                                   
EC                     54.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0       54.0   
EO                     54.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0       54.0   

                       ...   edges_std             edges_cv            \
                 mean  ...         75%         max    count      mean   
condition              ...                                              
EC         579.908807  ...  222.826503  335.894109     54.0  0.334855   
EO         593.000746  ...  224.296778  341.371652     54.0  0.339099   

                                                                       
                std       min       25%       50%       75%       max  
condition                                                              
EC         0.066382  0.232432  0.285072  0.333624  0.376101  0.531529  
EO         0.062419  0.222316  0.301142  0.332907  0.369830  0.537939  

[2 rows x 32 columns]

### Boxplots EO vs EC

Eficiência global

In [17]:
fig = px.box(
    df_global,
    x="condition",
    y="global_efficiency",
    color="condition",
    points="all",
    title="Eficiência Global da Rede por Condição Experimental"
)
fig.show()

Número médio de arestas

In [18]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_mean",
    color="condition",
    points="all",
    title="Número Médio de Arestas por Condição"
)

fig.show()


### Variabilidade da conectividade

In [19]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_cv",
    color="condition",
    points="all",
    title="Coeficiente de Variação da Conectividade ao Longo do Tempo"
)
fig.show()


### Correlação entre métricas globais

In [21]:
corr = df_global[GLOBAL_FEATURES].corr()
corr

,global_efficiency,edges_mean,edges_std,edges_cv
global_efficiency,NaN,NaN,NaN,NaN
edges_mean,NaN,1.000000,0.735345,-0.204254
edges_std,NaN,0.735345,1.000000,0.497826
edges_cv,NaN,-0.204254,0.497826,1.000000


In [22]:
fig = px.imshow(
    corr,
    text_auto=True,
    title="Matriz de Correlação entre Métricas Globais",
    color_continuous_scale="RdBu_r",
    aspect="auto"
)
fig.show()
